### Magic: The Gathering - Data Wrangling
* Physical Cards, English Only, Secondary Market Price - Retail

In [15]:
import pandas as pd

#### Clean card data.

In [ ]:
dfm = pd.read_csv('../data/dataMagic/cardsMagic.csv', low_memory = False) # 8/26/25

# Specify the needed columns
dfm = dfm[["availability", "colors", "language", "name", "rarity", "setCode", "types", "uuid"]]

# Replace all NaN with "C" for "Colorless"
dfm["colors"] = dfm["colors"].fillna("C")

# Specifying the rows to keep involving paper
dfm = dfm[
    (dfm["availability"] == "mtgo, paper") | 
    (dfm["availability"] == "paper") | 
    (dfm["availability"] == "arena, mtgo, paper") |
    (dfm["availability"] == "arena, paper") 
    ]

# Only want the English card versions
dfm = dfm[dfm["language"] == "English"]

# Remove the basic lands from each set.  
# These lands are printed for most sets in bulk and are mostly worthless, barring outliers.  
# This will tighten our dataset and focus it toward value.  
basic_lands = ["Forest", "Island", "Mountain", "Plains", "Swamp"]
dfm = dfm[~dfm["name"].isin(basic_lands)]

#### Add and merge sets csv.

In [17]:
dfmSets = pd.read_csv('../data/dataMagic/setsMagic.csv') # source date: 9/22/25

dfm2 = pd.merge(dfm, dfmSets, on = "setCode", how = "inner")

#### Add and merge prices csv.

In [ ]:
dfmPrices =  pd.read_csv('../data/dataMagic/pricesMagic.csv') # source date: 8/27/25

# Specify and remove the types we do not want in the dataframe.
mtgo = ["mtgo"]
buylist = ["buylist"]
cardmarket = ["cardmarket"]
dfmPrices = dfmPrices[~dfmPrices["gameAvailability"].isin(mtgo)]
dfmPrices = dfmPrices[~dfmPrices["providerListing"].isin(buylist)]
dfmPrices = dfmPrices[~dfmPrices["priceProvider"].isin(cardmarket)]

dfm3 = pd.merge(dfm2, dfmPrices, on = "uuid", how = "left")

#### dfm3 will be used for SQL queries and individual price lookups.

In [19]:
# Add a colum for average market price.
dfm3["avgMarketPrice"] = dfm3.groupby(['uuid', 'cardFinish'])['price'].transform('mean')

# Round to 2 decimal places.
dfm3["avgMarketPrice"] = dfm3["avgMarketPrice"].round(2)

# Availability is no longer needed, since we now have gameAvailability from prices csv.
dfm3.drop(columns = ["availability"], inplace = True)

# Convert release dates to datetime for plotting.
dfm3['releaseDate'] = pd.to_datetime(dfm3['releaseDate'], format = '%m/%d/%Y', errors = 'raise')

# Make a more viewer-friendly column and sort order.
newOrderM = ['name', 'setCode', 'setName', 'language', 'types', 'colors', 'rarity', 'cardFinish', 'releaseDate', 'releaseYear', 'gameAvailability', 
             'priceProvider', 'price', 'avgMarketPrice', 'currency', 'providerListing', 'date', 'uuid']
dfm3 = dfm3[newOrderM]

dfm3 = dfm3.sort_values(by=["releaseDate", "setName", "name"])

# Reset index after manipulation and to check new number of rows.
# Drop the original index column.
dfm3 = dfm3.reset_index(drop = True)

# Make a pickle file for easier/safer imports.
dfm3.to_pickle("dfm3.pkl")

#### dfm4 will be used for visualization and as a final, cleaner version.

In [20]:
# Individual prices and price providers not relevant for data viz.
dfm4 = dfm3.drop(columns = ["price", "priceProvider"])

# Remove dupes.
dfm4.drop_duplicates(keep = "first", inplace = True)

# Reset index again.
dfm4 = dfm4.reset_index(drop = True)

# Make a pickle file for easier/safer imports.
dfm4.to_pickle("dfm4.pkl")

# If needed as its own csv file, uncomment:
# dfm4.to_csv("../data/dataMagic/completeMagicClean.csv", index = False)